In [ ]:
# In Jupyter Notebook
import sys
import os

sys.path.append(r"C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/ML_Models")
from experiments.run_model import run_experiment

# Run experiment
run_experiment(r"C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/ML_Models/configs/lightgbm_config.yaml")

In [ ]:
# In Jupyter Notebook
import sys
import os

# Set project path
project_path = r"C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/ML_Models"

# Change working directory to project folder
os.chdir(project_path)

# Add to path
sys.path.append(project_path)

from experiments.run_model import run_experiment

# Now relative path works
run_experiment("configs/xgboost_config.yaml")

In [ ]:
# In Jupyter cell
%run predictions/run_prediction.py --model models/catboost_model.pkl --config configs/catboost_config.yaml --data "C:/Users/Phong/OneDrive - ICB Construction/Phong/data/Python_ETL/DS/Database/Jobs to be Predicted.xlsx" --output "predictions/catboost_jobs_results.xlsx"

In [2]:
# Load data
import pandas as pd

job_data_path = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Jobs Data.xlsx"

job_data_df = pd.read_excel(job_data_path)
job_data_df.describe()
job_data_df.columns


Index(['Job_Number', 'Job_Value', 'Contract_Establishment', 'Estimator',
       'Foreman', 'Job_Area', 'Main_Contractor', 'Suburb', 'Supervisor',
       'Job_Description', 'Saving_or_Loss', 'Timber_RW', 'RC_Piles',
       'Steel_Beam', 'Driven_Timber', 'Driven_Steel', 'Sheetpile', 'Anchors',
       'Blocks', 'Shotcrete', 'Capping_Beam', 'Walers', 'Earthworks',
       'Concrete_Slab', 'Precast', 'No_Wall_Types', 'Code',
       'Total_to_Date_Claimed', 'Amount_Increased', 'Adjusted_Savings'],
      dtype='object')

In [ ]:
# Linear Regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd

# Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and Train Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
Y_pred = model.predict(X_test)

# Print Coefficients
print(f"Intercept: {model.intercept_}")
print(f"Slope: {model.coef_}")

# Evaluate
print(f"R² Score: {r2_score(y_test, Y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, Y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, Y_pred):.4f}")

Z_pred = model.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - Without Pipeline and Tuning

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd

# Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create Polynomial features
degree = 2
poly = PolynomialFeatures(degree, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.fit_transform(X_test)

# Train model on polynomial features
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)

# Predict
Y_pred = model_poly.predict(X_test_poly)

# Print Coefficients
print(f'Intercept: {model_poly.intercept_}')
print(f'Coefficients: {model_poly.coef_}')


Z_pred = model.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - With Pipeline but no Tuning

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create conprehensive pipeline
pipeline = Pipeline([
    # Step 1: Handle missing values
    ('imputer', SimpleImputer(strategy='median')),
    # Step 2: Scale features
    ('scaler', StandardScaler()),
    # Step 3: Create polynomial features
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    # Step 4: Final Estimator
    ('model', LinearRegression())
])

# Train model 
pipeline.fit(X_train, y_train)

# Predict
Y_pred = pipeline.predict(X_test)

# Model performance
score = pipeline.score(X_test, y_test)
print(f'R2 Score: {score}')

# Get parameters
params = pipeline.get_params()
print(params['poly__degree'])  # Access nested parameters

# Print Coefficients
print(f'Intercept: {pipeline.named_steps["model"].intercept_}')
print(f'Coefficients shape: {pipeline.named_steps["model"].coef_.shape}')
print(f'Coefficients: {pipeline.named_steps["model"].coef_}')


Z_pred = pipeline.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - With Pipeline and Tuning

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1, Split data into features and target
X = job_data_df.drop(['Estimator',
                      'Foreman', 
                      'Job_Area', 
                      'Main_Contractor', 
                      'Suburb', 
                      'Supervisor',
                      'Job_Description',
                      'Saving_or_Loss'], 
                      axis=1)

y = job_data_df['Saving_or_Loss']

# 2, Train - Test data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3, Create pipeline with scaling and polynomial features + linear regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Important for polynomial features
    ('poly', PolynomialFeatures()),
    ('model', LinearRegression())
])

# 4, Define parameter grid
param_grid = {
    'poly__degree': [1, 2, 3, 4, 5],  # Try different polynomial degrees
    'poly__include_bias': [True, False],  # Include intercept term
    'poly__interaction_only': [True, False]  # Only interaction terms vs all terms
}

# 5, Perform grid search with cross-validation
grid_search = GridSearchCV(
    pipeline, 
    param_grid,
    cv=5,           # 5-fold cross-validation
    scoring='r2',   # Optimize for R-squared
    n_jobs=-1,      # Use all CPU cores
    verbose=1
)

print("performing Grid Search for best Polynomial parameter ...")

# 6, Train model
grid_search.fit(X_train, y_train)

  # Best parameters
print("\n" + "="*50)
print("BEST PARAMETERS FOUND:")
print("="*50)
print(f"Best Degree: {grid_search.best_params_['poly__degree']}")
print(f"Best include_bias: {grid_search.best_params_['poly__include_bias']}")
print(f"Best interaction_only: {grid_search.best_params_['poly__interaction_only']}")
print(f"Best CV R² Score: {grid_search.best_score_:.4f}")

  # Best model
best_poly_model = grid_search.best_estimator_

# 7, Predictions
y_pred = best_poly_model.predict(X_test)

# 8, Evaluate
print("\n" + "="*50)
print("MODEL EVALUATION ON TEST SET:")
print("="*50)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

# 9, Usage
Z_pred = model.predict([[10000, 2000, 3000, 40000]])
print(Z_pred)


In [ ]:
# Polynomial Regression - With Pipeline and Tuning and Encode for Categorical Features

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# 1, Define which columns are numeric and which are categorical
numeric_features = ['Job_Number', 'Job_Value', 'Total_Claimed', 'Total_Cost']
categorical_features = ['Estimator', 'Foreman', 'Supervisor']

# 2, Prepare the data
X = job_data_df[numeric_features + categorical_features]
y = job_data_df['Saving_or_Loss']

# 3, Data Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4, Create preprocessing for numeric columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 5, Create preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 6, Combine preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# 7, Create full pipeline with preprocessor, polu features and model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures()),
    ('model', Ridge())    
    # 'preprocessor__num__imputer__strategy': ['mean', 'median', 'most_frequent'] --For tuning imputer
])

# 8, Define parameter grid tuning
param_grid = {
    'poly__degree': [1, 2, 3, 4],
    #'poly__include_bias': [True, False],
    'poly__interaction_only': [True, False],
    'model__alpha': [0.1, 1.0, 10.0]  # Regularization strength
}

# 9, Perform grid search with cross-validation
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1    
)

print("Performing Grid Search for the best parameters...")
grid_search.fit(X_train, y_train)


# 10. Best parameters
print("\n" + "="*70)
print("BEST PARAMETERS FOUND:")
print("="*70)
print(f"Best Degree: {grid_search.best_params_['poly__degree']}")
# print(f"Best include_bias: {grid_search.best_params_['poly__include_bias']}")
print(f"Best interaction_only: {grid_search.best_params_['poly__interaction_only']}")
print(f"Best CV R² Score: {grid_search.best_score_:.4f}")

# 11. Best model
best_model = grid_search.best_estimator_

# 12. Predictions
y_pred = best_model.predict(X_test)

# 13. Evaluate
print("\n" + "="*70)
print("MODEL EVALUATION ON TEST SET:")
print("="*70)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

# 14. Get feature information
preprocessor_fitted = best_model.named_steps['preprocessor']
poly_fitted = best_model.named_steps['poly']

# Get feature names after preprocessing
feature_names = []
for name, transformer, columns in preprocessor_fitted.transformers_:
    if name == 'num':
        feature_names.extend(columns)
    elif name == 'cat':
        # Get one-hot encoded feature names
        encoder = transformer.named_steps['onehot']
        feature_names.extend(encoder.get_feature_names_out(columns))

# Get polynomial feature names
poly_feature_names = poly_fitted.get_feature_names_out(feature_names)
print(f"\nTotal features after polynomial transformation: {len(poly_feature_names)}")
print(f"First 10 polynomial features: {poly_feature_names[:10]}")

# 15. Make predictions for new data WITH categorical variables
# Create new job data with both numeric and categorical values
new_job = pd.DataFrame({
    'Job_Number': [10003],
    'Job_Value': [300000], 
    'Total_Claimed': [200000], 
    'Total_Cost': [150000],
    'Estimator': ['John Doe'],  # Categorical
    'Foreman': ['Jane Smith'],  # Categorical
    'Supervisor': ['Bob Johnson']  # Categorical
})

Z_pred = best_model.predict(new_job)
print(f"\nPrediction for new job: ${Z_pred[0]:,.2f}")

#Categorical + Polynomial = Problems - One-hot encoded columns create sparse matrices that blow up with polynomial features
#Use interaction_only=True - This avoids x² terms which cause the most issues


In [ ]:
# Regularized Polynomial Regression

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
X = job_data_df.drop(['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 
                      'Suburb', 'Supervisor', 'Job_Description', 'Saving_or_Loss'], axis=1)
y = job_data_df['Saving_or_Loss']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create base pipeline (without regularization for comparison)
base_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])

# Ridge pipeline
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('ridge', Ridge(alpha=1.0))
])

# Lasso pipeline
lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lasso', Lasso(alpha=1.0, max_iter=10000))
])

# ElasticNet pipeline
elastic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('elastic', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))
])

# Train all models
base_pipeline.fit(X_train, y_train)
ridge_pipeline.fit(X_train, y_train)
lasso_pipeline.fit(X_train, y_train)
elastic_pipeline.fit(X_train, y_train)

# Compare results
print("="*60)
print("MODEL COMPARISON (Without Hyperparameter Tuning)")
print("="*60)

models = {
    'Linear Regression (No Reg)': base_pipeline,
    'Ridge (L2)': ridge_pipeline,
    'Lasso (L1)': lasso_pipeline,
    'ElasticNet (L1+L2)': elastic_pipeline
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    # Get coefficient statistics
    if 'ridge' in model.named_steps:
        coef = model.named_steps['ridge'].coef_
    elif 'lasso' in model.named_steps:
        coef = model.named_steps['lasso'].coef_
    elif 'elastic' in model.named_steps:
        coef = model.named_steps['elastic'].coef_
    else:
        coef = model.named_steps['model'].coef_
    
    print(f"\n{name}:")
    print(f"  R² Score: {r2:.4f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  Non-zero coeffs: {np.sum(coef != 0)} / {len(coef)}")
    print(f"  Max coefficient: {np.abs(coef).max():.2f}")
    print(f"  Mean coefficient: {np.abs(coef).mean():.2f}")

In [ ]:
# Regularized Polynomial Regression - Tuning

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
X = job_data_df.drop(['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 
                      'Suburb', 'Supervisor', 'Job_Description', 'Saving_or_Loss'], axis=1)
y = job_data_df['Saving_or_Loss']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ridge pipeline
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('ridge', Ridge(alpha=1.0))
])

# Lasso pipeline
lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lasso', Lasso(alpha=1.0, max_iter=10000))
])

# ElasticNet pipeline
elastic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('elastic', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))
])

# Ridge Hyperparameter Tuning
print("\n" + "="*60)
print("RIDGE REGRESSION TUNING")
print("="*60)

ridge_param_grid = {
    'poly__degree': [1, 2, 3, 4],
    'ridge__alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'ridge__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag']
}

ridge_search = GridSearchCV(
    ridge_pipeline, 
    ridge_param_grid, 
    cv=5, 
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
ridge_search.fit(X_train, y_train)

print(f"Best Ridge Parameters: {ridge_search.best_params_}")
print(f"Best Ridge CV R²: {ridge_search.best_score_:.4f}")
print(f"Ridge Test R²: {ridge_search.score(X_test, y_test):.4f}")

# Lasso Hyperparameter Tuning
print("\n" + "="*60)
print("LASSO REGRESSION TUNING")
print("="*60)

lasso_param_grid = {
    'poly__degree': [1, 2, 3],
    'lasso__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10],
    'lasso__selection': ['cyclic', 'random']
}

lasso_search = GridSearchCV(
    lasso_pipeline, 
    lasso_param_grid, 
    cv=5, 
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
lasso_search.fit(X_train, y_train)

print(f"Best Lasso Parameters: {lasso_search.best_params_}")
print(f"Best Lasso CV R²: {lasso_search.best_score_:.4f}")
print(f"Lasso Test R²: {lasso_search.score(X_test, y_test):.4f}")

# Count features selected by Lasso
best_lasso = lasso_search.best_estimator_
lasso_coef = best_lasso.named_steps['lasso'].coef_
print(f"Features selected by Lasso: {np.sum(lasso_coef != 0)} / {len(lasso_coef)}")

# ElasticNet Hyperparameter Tuning
print("\n" + "="*60)
print("ELASTICNET REGRESSION TUNING")
print("="*60)

elastic_param_grid = {
    'poly__degree': [1, 2, 3],
    'elastic__alpha': [0.001, 0.01, 0.1, 1, 10],
    'elastic__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],  # 0=Ridge, 1=Lasso
    'elastic__selection': ['cyclic', 'random']
}

elastic_search = GridSearchCV(
    elastic_pipeline, 
    elastic_param_grid, 
    cv=5, 
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
elastic_search.fit(X_train, y_train)

print(f"Best ElasticNet Parameters: {elastic_search.best_params_}")
print(f"Best ElasticNet CV R²: {elastic_search.best_score_:.4f}")
print(f"ElasticNet Test R²: {elastic_search.score(X_test, y_test):.4f}")

# Compare all tuned models
print("\n" + "="*60)
print("FINAL COMPARISON (Tuned Models)")
print("="*60)

tuned_models = {
    'Ridge (Tuned)': ridge_search,
    'Lasso (Tuned)': lasso_search,
    'ElasticNet (Tuned)': elastic_search
}

best_r2 = -np.inf
best_name = None
best_model = None

for name, model in tuned_models.items():
    test_r2 = model.score(X_test, y_test)
    print(f"\n{name}:")
    print(f"  Test R²: {test_r2:.4f}")
    print(f"  Best Params: {model.best_params_}")
    
    if test_r2 > best_r2:
        best_r2 = test_r2
        best_name = name
        best_model = model

print(f"\n{'='*60}")
print(f"🏆 BEST MODEL: {best_name} with R² = {best_r2:.4f}")
print("="*60)

In [ ]:
# Visualize the effect of alpha on coefficients
import matplotlib.pyplot as plt

alphas = [0.001, 0.01, 0.1, 1, 10, 100]
coefficient_paths = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('ridge', ridge)
    ])
    ridge_pipeline.fit(X_train, y_train)
    coefs = ridge_pipeline.named_steps['ridge'].coef_
    coefficient_paths.append(coefs)

coefficient_paths = np.array(coefficient_paths)

# Plot coefficient paths
plt.figure(figsize=(10, 6))
for i in range(min(10, coefficient_paths.shape[1])):  # Plot first 10 coefficients
    plt.plot(alphas, coefficient_paths[:, i], label=f'Feature {i}')

plt.xscale('log')
plt.xlabel('Alpha (regularization strength)')
plt.ylabel('Coefficient Value')
plt.title('Ridge Coefficient Paths\n(As alpha increases, coefficients shrink toward zero)')
plt.legend(loc='best', ncol=2, fontsize=8)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Decision Tree Regression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1, Define which columns are numeric and which are categorical
binary_job_type_feature = [
    'Timber_RW', 'RC_Piles', 'Steel_Beam', 'Driven_Timber',	'Driven_Steel',
    'Sheetpile', 'Anchors',	'Blocks', 'Shotcrete', 'Capping_Beam', 'Walers',
    'Earthworks', 'Concrete_Slab',	'Precast'
]
numeric_features = ['Job_Number', 'Job_Value', 'Contract_Establishment', 'No_Wall_Types']
categorical_features = ['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 'Suburb', 'Supervisor']

# 2, Prepare the data
X = job_data_df[numeric_features + categorical_features + binary_job_type_feature]
y = job_data_df['Adjusted_Savings']

# 3, Data Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4, Create preprocessing for numeric and binary columns
binary_transformer = SimpleImputer(strategy='constant', fill_value=0)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# 5, Create preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 6, Combined preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', binary_transformer, binary_job_type_feature),
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# 7. Create the pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dt', DecisionTreeRegressor(random_state=42))
])
    
# 8, Param Grid
param_grid = {
    'dt__max_depth': [8, 10, 12, 15],  # Moderate depths
    'dt__min_samples_split': [5, 10, 15, 20],  # Moderate splits
    'dt__min_samples_leaf': [2, 3, 5, 8],  # Moderate leaf sizes
    'dt__max_features': ['sqrt', 'log2'],  # Not None
    'dt__criterion': ['squared_error', 'absolute_error'],
    'dt__min_impurity_decrease': [0.0, 0.001, 0.005, 0.01],
    'dt__ccp_alpha': [0.0, 0.0001, 0.001, 0.005, 0.01]  # Small pruning values
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print("Performing Grid Search for Decision Tree...")
grid_search.fit(X_train, y_train)

print("\n" + "="*60)
print("BEST DECISION TREE PARAMETERS")
print("="*60)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")
print(f"\nBest CV R²: {grid_search.best_score_:.4f}")

# 10, Best model
best_dt = grid_search.best_estimator_
y_pred = best_dt.predict(X_test)

# 11, Evaluate
print("="*60)
print("DECISION TREE WITH CATEGORICAL FEATURES")
print("="*60)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")

# 12. Baseline comparison
y_test_mean = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, y_test_mean)
baseline_r2 = r2_score(y_test, y_test_mean)

print("\n" + "="*60)
print("BASELINE COMPARISON")
print("="*60)
print(f"Baseline (predict mean) MAE: {baseline_mae:.2f}")
print(f"Model MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Improvement: {((baseline_mae - mean_absolute_error(y_test, y_pred)) / baseline_mae * 100):.1f}%")

if mean_absolute_error(y_test, y_pred) < baseline_mae:
    print("✅ Model learned useful patterns (better than baseline)")
else:
    print("❌ Model failed to learn (worse than baseline)")



'''
# 8. Find optimal alpha for pruning (FIXED: need to fit on processed data)
# First, preprocess the data to find ccp_alphas
X_train_processed = preprocessor.fit_transform(X_train)
path = DecisionTreeRegressor().cost_complexity_pruning_path(X_train_processed, y_train)
ccp_alphas = path.ccp_alphas
# Take only reasonable alphas (avoid too many or too large)
ccp_alphas = ccp_alphas[ccp_alphas < 0.1]  # Limit alpha range
if len(ccp_alphas) > 20:  # Limit number of alphas
    ccp_alphas = np.linspace(ccp_alphas[0], ccp_alphas[-1], 20)
'''


In [ ]:
# Decision Tree Regression - With RandomizedSearchCV

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1. Define which columns are numeric, binary, and categorical
binary_job_type_feature = [
    'Timber_RW', 'RC_Piles', 'Steel_Beam', 'Driven_Timber', 'Driven_Steel',
    'Sheetpile', 'Anchors', 'Blocks', 'Shotcrete', 'Capping_Beam', 'Walers',
    'Earthworks', 'Concrete_Slab', 'Precast'
]

numeric_features = ['Job_Number', 'Job_Value', 'Contract_Establishment', 'No_Wall_Types']
categorical_features = ['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 'Suburb', 'Supervisor']

# 2. Prepare the data
X = job_data_df[numeric_features + categorical_features + binary_job_type_feature]
y = job_data_df['Adjusted_Savings']

# 3. Data Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check for missing values
missing_counts = X_train.isnull().sum()
if missing_counts.sum() > 0:
    print("\nColumns with missing values:")
    print(missing_counts[missing_counts > 0])
else:
    print("\n✅ No missing values found in training data")

# 4. Create preprocessing for numeric and binary columns
# Binary features - just impute with 0 (no scaling needed for trees)
binary_transformer = SimpleImputer(strategy='constant', fill_value=0)

# Numeric features - impute with median (no scaling needed for trees)
numeric_transformer = SimpleImputer(strategy='median')

# 5. Create preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 6. Combined preprocessor - FIXED: Added comma between transformers!
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', binary_transformer, binary_job_type_feature),  
        ('num', numeric_transformer, numeric_features),           
        ('cat', categorical_transformer, categorical_features)  
    ]
)

# 7. Create the pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dt', DecisionTreeRegressor(random_state=42))
])

# 8. Param Grid - Balanced for your data
param_grid = {
    'dt__max_depth': [8, 10, 12, 15],  # Moderate depths
    'dt__min_samples_split': [5, 10, 15, 20],  # Moderate splits
    'dt__min_samples_leaf': [2, 3, 5, 8],  # Moderate leaf sizes
    'dt__max_features': ['sqrt', 'log2'],  # Not None for randomness
    'dt__criterion': ['squared_error', 'absolute_error'],
    'dt__min_impurity_decrease': [0.0, 0.001, 0.005, 0.01],
    'dt__ccp_alpha': [0.0, 0.0001, 0.001, 0.005, 0.01]  # Small pruning values
}

# Calculate total combinations
total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nTotal parameter combinations: {total_combinations:,}")

# Reduce grid if too large
if total_combinations > 1000:
    print("⚠️ Large grid detected. Consider reducing parameters for faster search.")
    # Option: Use RandomizedSearchCV instead
    print("Using RandomizedSearchCV for efficiency...")
    
    grid_search = RandomizedSearchCV(
        pipeline,
        param_grid,
        n_iter=100,  # Try 100 random combinations
        cv=5,
        scoring='r2',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
else:
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=1
    )

print("\nPerforming Grid Search for Decision Tree...")
grid_search.fit(X_train, y_train)

# 9. Best parameters
print("\n" + "="*60)
print("BEST DECISION TREE PARAMETERS")
print("="*60)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")
print(f"\nBest CV R²: {grid_search.best_score_:.4f}")

# 10. Best model
best_dt = grid_search.best_estimator_
y_pred = best_dt.predict(X_test)

# 11. Evaluate
print("\n" + "="*60)
print("DECISION TREE PERFORMANCE")
print("="*60)
test_r2 = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)

print(f"R² Score: {test_r2:.4f}")
print(f"RMSE: ${test_rmse:,.2f}")
print(f"MAE: ${test_mae:,.2f}")

# 12. Baseline comparison
y_test_mean = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, y_test_mean)
baseline_r2 = r2_score(y_test, y_test_mean)

print("\n" + "="*60)
print("BASELINE COMPARISON")
print("="*60)
print(f"Baseline (predict mean) MAE: ${baseline_mae:,.2f}")
print(f"Model MAE: ${test_mae:,.2f}")
print(f"Improvement: {((baseline_mae - test_mae) / baseline_mae * 100):.1f}%")

if test_mae < baseline_mae:
    print("✅ Model learned useful patterns (better than baseline)")
else:
    print("❌ Model failed to learn (worse than baseline)")

# 13. Overfitting check
train_pred = best_dt.predict(X_train)
train_r2 = r2_score(y_train, train_pred)
overfit_gap = train_r2 - test_r2

print("\n" + "="*60)
print("OVERFITTING ANALYSIS")
print("="*60)
print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Overfitting Gap: {overfit_gap:.4f}")

if overfit_gap > 0.2:
    print("⚠️ HIGH OVERFITTING - Model is too complex")
    print("   Consider: reducing max_depth, increasing min_samples_split")
elif overfit_gap > 0.1:
    print("⚠️ MODERATE OVERFITTING - Some regularization needed")
else:
    print("✅ Good generalization - Model complexity is appropriate")

# 14. Cross-validation score
cv_scores = cross_val_score(best_dt, X_train, y_train, cv=5, scoring='r2')
print(f"\nCross-validation R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# 15. Feature importance
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

# Get feature names after preprocessing
feature_names = []
for name, transformer, columns in best_dt.named_steps['preprocessor'].transformers_:
    if name == 'binary':
        feature_names.extend(columns)
    elif name == 'num':
        feature_names.extend(columns)
    elif name == 'cat':
        # Get one-hot encoded feature names
        encoder = transformer.named_steps['onehot']
        feature_names.extend(encoder.get_feature_names_out(columns))

# Get importance
importances = best_dt.named_steps['dt'].feature_importances_

# Create and sort importance DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Top 15 most important features:")
print(importance_df.head(15).to_string(index=False))

# 16. Tree statistics
tree_depth = best_dt.named_steps['dt'].get_depth()
tree_nodes = best_dt.named_steps['dt'].tree_.node_count
tree_leaves = best_dt.named_steps['dt'].get_n_leaves()

print(f"\nTree Statistics:")
print(f"  Depth: {tree_depth}")
print(f"  Number of nodes: {tree_nodes}")
print(f"  Number of leaves: {tree_leaves}")


✅ No missing values found in training data

Total parameter combinations: 5,120
⚠️ Large grid detected. Consider reducing parameters for faster search.
Using RandomizedSearchCV for efficiency...

Performing Grid Search for Decision Tree...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

BEST DECISION TREE PARAMETERS
dt__min_samples_split: 5
dt__min_samples_leaf: 5
dt__min_impurity_decrease: 0.01
dt__max_features: sqrt
dt__max_depth: 10
dt__criterion: absolute_error
dt__ccp_alpha: 0.0001

Best CV R²: 0.0026

DECISION TREE PERFORMANCE
R² Score: 0.1461
RMSE: $78,631.74
MAE: $38,222.69

BASELINE COMPARISON
Baseline (predict mean) MAE: $41,533.61
Model MAE: $38,222.69
Improvement: 8.0%
✅ Model learned useful patterns (better than baseline)

OVERFITTING ANALYSIS
Train R²: 0.0364
Test R²: 0.1461
Overfitting Gap: -0.1097
✅ Good generalization - Model complexity is appropriate

Cross-validation R²: 0.0026 (+/- 0.0662)

FEATURE IMPORTANCE
Top 15 most important features:
        

In [5]:
# Random Forest Regression

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ===============================================================================================
# Feature Engineering
# ===============================================================================================
# 1, Create more meaningful features from existing ones
def engineer_features(df):
    """Create additional features to improve predictive power"""
    df_eng = df.copy()
        
    # 1.1 Project Required Drill
    df_eng['Required_Drill_Job'] = (
        (df_eng['Steel_Beam'] == 1) | 
        (df_eng['RC_Piles'] == 1) | 
        (df_eng['Timber_RW'] == 1)
    ).astype(int)

    # 1.2 Project Required Attachments
    df_eng['Required_Attachments_Job'] = (
        (df_eng['Driven_Timber'] == 1) | 
        (df_eng['Driven_Steel'] == 1) | 
        (df_eng['Sheetpile'] == 1)
    ).astype(int)
        
    # 1.3 Establishment and Total Value Ratio
    df_eng['Est_to_Total_Value_Ratio'] = df_eng['Contract_Establishment'] / df_eng['Job_Value']
    
    # 1.4 Project size category
    df_eng['Job_Value_Category'] = pd.cut(
        df_eng['Job_Value'],
        bins=[0, 100000, 500000, 1000000, float('inf')],
        labels=['Small', 'Medium', 'Large', 'Very Large']
    )
    
    return df_eng

# Apply feature engineering
job_data_df_eng = engineer_features(job_data_df)

# 2, Group rare categories into 'Other' to prevent overfitting
def group_rare_categories(df, column, min_frequency=0.05):
    """
    Group rare categories into 'Other' to prevent overfitting
    """
    # Calculate frequency
    freq = df[column].value_counts(normalize=True) # normalize=true to return proportions of count values
    
    # Identify rare categories
    rare_categories = freq[freq < min_frequency].index.tolist() # create a list of all rare categories
    
    # Group them
    df_clean = df.copy()
    df_clean[column] = df_clean[column].apply(
        lambda x: 'Other' if x in rare_categories else x
    )
    
    print(f"{column}: Grouped {len(rare_categories)} rare categories into 'Other'")
    return df_clean

# Apply to categorical columns
categorical_features_to_group = ['Estimator', 'Foreman', 'Main_Contractor', 'Supervisor', 'Suburb']

for col in categorical_features_to_group:
    job_data_df_eng = group_rare_categories(job_data_df_eng, col, min_frequency=0.03)

# ================================================================================================
# Model Training
# ================================================================================================

# 1. Define which columns are numeric, binary, and categorical
binary_job_type_feature = [
    'Timber_RW', 'RC_Piles', 'Steel_Beam', 'Driven_Timber', 'Driven_Steel',
    'Sheetpile', 'Anchors', 'Blocks', 'Shotcrete', 'Capping_Beam', 'Walers',
    'Earthworks', 'Concrete_Slab', 'Precast', 'Required_Attachments_Job', 'Required_Drill_Job'
]

numeric_features = ['Job_Number', 'Job_Value', 'Contract_Establishment', 'No_Wall_Types', 'Est_to_Total_Value_Ratio']
categorical_features = ['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 'Suburb', 'Supervisor', 'Job_Value_Category']

# 2. Prepare the data
X = job_data_df_eng[numeric_features + categorical_features + binary_job_type_feature]
y = job_data_df_eng['Adjusted_Savings']

# 3. Data Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check for missing values
missing_counts = X_train.isnull().sum()
if missing_counts.sum() > 0:
    print("\nColumns with missing values:")
    print(missing_counts[missing_counts > 0])
else:
    print("\n✅ No missing values found in training data")

# 4 Create preprocessing for numeric and binary columns
binary_transformer = SimpleImputer(strategy='constant', fill_value=0)
numeric_transformer = SimpleImputer(strategy='median')

# 5 Create prprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 6 Combine preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', binary_transformer, binary_job_type_feature),  
        ('num', numeric_transformer, numeric_features),           
        ('cat', categorical_transformer, categorical_features) 
    ]
)

# 7 Create Pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(random_state=42))
])

# 8 Param Grid
rf_param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [10, 15, None],
    'rf__min_samples_split': [5, 10, 20],
    'rf__min_samples_leaf': [2, 5, 10],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__bootstrap': [True]
}

rf_search = RandomizedSearchCV(
    rf_pipeline, 
    rf_param_grid, 
    n_iter=100,
    cv=5, 
    scoring='r2', 
    n_jobs=-1,
    random_state=42,
    verbose=1
)   

rf_search.fit(X_train, y_train)

# 9 Best parameters
print("\n" + "="*60)
print("Best Random Forest Tree Parameters")
print("="*60)
for param, value in rf_search.best_params_.items():
    print(f"{param}: {value}")
print(f"\nBest CV R2: {rf_search.best_score_:.4f}")

# 10 Best model
best_rf = rf_search.best_estimator_
y_pred = best_rf.predict(X_test)

# 11. Evaluate
print("\n" + "="*60)
print("DECISION TREE PERFORMANCE")
print("="*60)
test_r2 = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)

print(f"R² Score: {test_r2:.4f}")
print(f"RMSE: ${test_rmse:,.2f}")
print(f"MAE: ${test_mae:,.2f}")

# 12. Baseline comparison
y_test_mean = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, y_test_mean)
baseline_r2 = r2_score(y_test, y_test_mean)

print("\n" + "="*60)
print("BASELINE COMPARISON")
print("="*60)
print(f"Baseline (predict mean) MAE: ${baseline_mae:,.2f}")
print(f"Model MAE: ${test_mae:,.2f}")
print(f"Improvement: {((baseline_mae - test_mae) / baseline_mae * 100):.1f}%")

if test_mae < baseline_mae:
    print("✅ Model learned useful patterns (better than baseline)")
else:
    print("❌ Model failed to learn (worse than baseline)")

# 13. Cross-validation score
cv_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='r2')
print(f"\nCross-validation R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# 14. Feature importance
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

# Get feature names after preprocessing
feature_names = []
for name, transformer, columns in best_rf.named_steps['preprocessor'].transformers_:
    if name == 'binary':
        feature_names.extend(columns)
    elif name == 'num':
        feature_names.extend(columns)
    elif name == 'cat':
        # Get one-hot encoded feature names
        encoder = transformer.named_steps['onehot']
        feature_names.extend(encoder.get_feature_names_out(columns))

# Get importance
importances = best_rf.named_steps['rf'].feature_importances_

# Create and sort importance DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Top 15 most important features:")
print(importance_df.head(15).to_string(index=False))

Estimator: Grouped 0 rare categories into 'Other'
Foreman: Grouped 0 rare categories into 'Other'
Main_Contractor: Grouped 0 rare categories into 'Other'
Supervisor: Grouped 0 rare categories into 'Other'
Suburb: Grouped 0 rare categories into 'Other'

✅ No missing values found in training data
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best Random Forest Tree Parameters
rf__n_estimators: 50
rf__min_samples_split: 20
rf__min_samples_leaf: 2
rf__max_features: sqrt
rf__max_depth: 10
rf__bootstrap: True

Best CV R2: 0.0540

DECISION TREE PERFORMANCE
R² Score: 0.0657
RMSE: $82,250.28
MAE: $43,221.64

BASELINE COMPARISON
Baseline (predict mean) MAE: $41,533.61
Model MAE: $43,221.64
Improvement: -4.1%
❌ Model failed to learn (worse than baseline)

Cross-validation R²: 0.0540 (+/- 0.1728)

FEATURE IMPORTANCE
Top 15 most important features:
                      feature  importance
                    Job_Value    0.315212
     Est_to_Total_Value_Ratio    0.131792
        

In [ ]:
# Create more meaningful features from existing ones

def engineer_features(df):
    """Create additional features to improve predictive power"""
    
    df_eng = df.copy()
        
    # 1. Project Required Drill
    df_eng['Required_Drill_Job'] = (
        (df_eng['Steel_Beam'] == 1) | 
        (df_eng['RC_Piles'] == 1) | 
        (df_eng['Timber_RW'] == 1)
    ).astype(int)

    # 2. Project Required Attachments
    df_eng['Required_Attachments_Job'] = (
        (df_eng['Driven_Timber'] == 1) | 
        (df_eng['Driven_Steel'] == 1) | 
        (df_eng['Sheetpile'] == 1)
    ).astype(int)
        
    # 3. Establishment and Total Value Ratio
    df_eng['Est_to_Total_Value_Ratio'] = df_eng['Contract_Establishment'] / df_eng['Job_Value']
    
    # 4. Project size category
    df_eng['Job_Value_Category'] = pd.cut(
        df_eng['Job_Value'],
        bins=[0, 100000, 500000, 1000000, float('inf')],
        labels=['Small', 'Medium', 'Large', 'Very Large']
    )
    
    return df_eng

# Apply feature engineering
job_data_df_eng = engineer_features(job_data_df)

job_data_df_eng.head()

In [ ]:
# Group rare categories into Other to prevent overfitting

def group_rare_categories(df, column, min_frequency=0.05):
    """
    Group rare categories into 'Other' to prevent overfitting
    """
    # Calculate frequency
    freq = df[column].value_counts(normalize=True) # normalize=true to return proportions of count values
    
    # Identify rare categories
    rare_categories = freq[freq < min_frequency].index.tolist() # create a list of all rare categories
    
    # Group them
    df_clean = df.copy()
    df_clean[column] = df_clean[column].apply(
        lambda x: 'Other' if x in rare_categories else x
    )
    
    print(f"{column}: Grouped {len(rare_categories)} rare categories into 'Other'")
    return df_clean

# Apply to categorical columns
categorical_features_to_group = ['Estimator', 'Foreman', 'Main_Contractor', 'Supervisor', 'Suburb']

for col in categorical_features_to_group:
    job_data_df = group_rare_categories(job_data_df, col, min_frequency=0.03)

job_data_df.head(20)

In [19]:
# ✅ # Random Forest Regression - CORRECT - No Data Leakage

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ================================================================================================
# Custom Transformers for Feature Engineering
# ================================================================================================

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer for feature engineering
    """
    def __init__(self):
        # Stores binary feature names
        self.binary_cols = [
            'Timber_RW', 'RC_Piles', 'Steel_Beam', 'Driven_Timber', 'Driven_Steel',
            'Sheetpile', 'Anchors', 'Blocks', 'Shotcrete', 'Capping_Beam', 'Walers',
            'Earthworks', 'Concrete_Slab', 'Precast'
        ]
        # Stores numeric feature names
        self.numeric_cols = ['Contract_Establishment', 'Job_Value']
        self.feature_names_ = None # Create an attribute
        
    def fit(self, X, y=None):
        # Just return self - no fitting needed
        return self
    
    def transform(self, X):
        X_eng = X.copy()
        
        # 1. Required Drill Job
        X_eng['Required_Drill_Job'] = (
            (X_eng['Steel_Beam'] == 1) | 
            (X_eng['RC_Piles'] == 1) | 
            (X_eng['Timber_RW'] == 1)
        ).astype(int)
        
        # 2. Required Attachments Job
        X_eng['Required_Attachments_Job'] = (
            (X_eng['Driven_Timber'] == 1) | 
            (X_eng['Driven_Steel'] == 1) | 
            (X_eng['Sheetpile'] == 1)
        ).astype(int)
        
        # 3. Establishment to Total Value Ratio
        X_eng['Est_to_Total_Value_Ratio'] = X_eng['Contract_Establishment'] / X_eng['Job_Value']
        
        # 4. Total Material Types
        X_eng['Total_Material_Types'] = X_eng[self.binary_cols].sum(axis=1)
        
        # 5. Complex Structure
        X_eng['Has_Complex_Structure'] = (
            (X_eng['Steel_Beam'] == 1) & 
            (X_eng['Concrete_Slab'] == 1) & 
            (X_eng['RC_Piles'] == 1)
        ).astype(int)
        
        # 6. Job Value Category (binning)
        X_eng['Job_Value_Category'] = pd.cut(
            X_eng['Job_Value'],
            bins=[0, 100000, 500000, 1000000, float('inf')],
            labels=['Small', 'Medium', 'Large', 'Very Large']
        )
        
        return X_eng

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """
    Group rare categories in categorical columns
    """
    def __init__(self, columns, min_frequency=0.03):
        self.columns = columns
        self.min_frequency = min_frequency
        self.rare_categories_ = {}  # attribute, initial empty, fill after fitting
        
    def fit(self, X, y=None):
        # Identify rare categories using ONLY training data
        for col in self.columns:
            if col in X.columns:
                freq = X[col].value_counts(normalize=True)
                rare = freq[freq < self.min_frequency].index.tolist()
                self.rare_categories_[col] = rare
        return self  # store back to self.rare_categories_
    
    def transform(self, X):
        X_clean = X.copy()
        for col in self.columns:
            if col in X_clean.columns and col in self.rare_categories_:
                rare = self.rare_categories_[col]
                X_clean[col] = X_clean[col].apply(
                    lambda x: 'Other' if x in rare else x
                )
        return X_clean

# ================================================================================================
# Complete Pipeline with Feature Engineering
# ================================================================================================

# 1. Define columns
binary_job_type_feature = [
    'Timber_RW', 'RC_Piles', 'Steel_Beam', 'Driven_Timber', 'Driven_Steel',
    'Sheetpile', 'Anchors', 'Blocks', 'Shotcrete', 'Capping_Beam', 'Walers',
    'Earthworks', 'Concrete_Slab', 'Precast'
]

numeric_features = ['Job_Value', 'Contract_Establishment', 'No_Wall_Types']
categorical_features = ['Estimator', 'Foreman', 'Job_Area', 'Main_Contractor', 'Suburb', 'Supervisor']

# 2. Prepare data (NO feature engineering here!)
X = job_data_df[numeric_features + categorical_features + binary_job_type_feature]
y = job_data_df['Adjusted_Savings']

# 3. Split FIRST - this is critical!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Create preprocessing pipeline
# Preprocessing for binary columns
binary_transformer = SimpleImputer(strategy='constant', fill_value=0)

# Preprocessing for numeric columns
numeric_transformer = SimpleImputer(strategy='median')

# Preprocessing for categorical columns
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 5. Create ColumnTransformer
preprocessor = ColumnTransformer([
    ('binary', binary_transformer, binary_job_type_feature),
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 6. Create FULL pipeline with feature engineering

rf_pipeline = Pipeline([
    ('feature_engineering', FeatureEngineer()),  # Custom transformer
    ('rare_category_grouper', RareCategoryGrouper(
        columns=['Estimator', 'Foreman', 'Main_Contractor', 'Supervisor', 'Suburb'],
        min_frequency=0.03
    )),
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# 8 Param Grid
rf_param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [10, 15, None],
    'rf__min_samples_split': [5, 10, 20],
    'rf__min_samples_leaf': [2, 5, 10],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__bootstrap': [True]
}

rf_search = RandomizedSearchCV(
    rf_pipeline, 
    rf_param_grid, 
    n_iter=100,
    cv=5, 
    scoring='r2', 
    n_jobs=-1,
    random_state=42,
    verbose=1
)   

# 7. Train (NO data leakage!)
print("Training with NO data leakage...")
rf_search.fit(X_train, y_train)
# 9 Best parameters
print("\n" + "="*60)
print("Best Random Forest Tree Parameters")
print("="*60)
for param, value in rf_search.best_params_.items():
    print(f"{param}: {value}")
print(f"\nBest CV R2: {rf_search.best_score_:.4f}")

# 10 Best model
best_rf = rf_search.best_estimator_
y_pred = best_rf.predict(X_test)

# 11. Evaluate
print("\n" + "="*60)
print("DECISION TREE PERFORMANCE")
print("="*60)
test_r2 = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)

print(f"R² Score: {test_r2:.4f}")
print(f"RMSE: ${test_rmse:,.2f}")
print(f"MAE: ${test_mae:,.2f}")

# 12. Baseline comparison
y_test_mean = np.full(len(y_test), y_train.mean())
baseline_mae = mean_absolute_error(y_test, y_test_mean)
baseline_r2 = r2_score(y_test, y_test_mean)

print("\n" + "="*60)
print("BASELINE COMPARISON")
print("="*60)
print(f"Baseline (predict mean) MAE: ${baseline_mae:,.2f}")
print(f"Model MAE: ${test_mae:,.2f}")
print(f"Improvement: {((baseline_mae - test_mae) / baseline_mae * 100):.1f}%")

if test_mae < baseline_mae:
    print("✅ Model learned useful patterns (better than baseline)")
else:
    print("❌ Model failed to learn (worse than baseline)")

Training with NO data leakage...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best Random Forest Tree Parameters
rf__n_estimators: 200
rf__min_samples_split: 20
rf__min_samples_leaf: 2
rf__max_features: sqrt
rf__max_depth: 15
rf__bootstrap: True

Best CV R2: 0.0142

DECISION TREE PERFORMANCE
R² Score: 0.0862
RMSE: $81,342.70
MAE: $42,296.67

BASELINE COMPARISON
Baseline (predict mean) MAE: $41,533.61
Model MAE: $42,296.67
Improvement: -1.8%
❌ Model failed to learn (worse than baseline)


In [ ]:
# Make prediction from data file

import pandas as pd
import joblib

# Load test data
test_path = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Jobs Data - Test.xlsx"
df_test = pd.read_excel(test_path)

# Split X (same features as training)
X_test = df_test[numeric_features + categorical_features + binary_job_type_feature]

# Load trained model (IMPORTANT: pipeline object)
model = best_rf   # or joblib.load("rf_model.pkl")

# Predict
predictions = model.predict(X_test)

df_test["Predictions"] = predictions

print(df_test.head(9))

In [22]:
import numpy as np
import pandas as pd

# ================================================================================================
# REAL EXAMPLE: Predicting Construction Savings
# ================================================================================================

# Sample data (simplified)
data = pd.DataFrame({
    'Job_Value': [100000, 200000, 150000, 250000, 180000],
    'Labor_Hours': [1000, 1500, 1200, 1800, 1400],
    'Actual_Saving': [10000, 15000, 12000, 18000, 14000]
})

print("Training Data:")
print(data)

# ================================================================================================
# STEP 1: Initial Prediction (F₀)
# ================================================================================================

# In XGBoost, initial prediction is the mean
F0 = data['Actual_Saving'].mean()
print(f"\nStep 1: Initial Prediction F₀ = {F0:.2f}")

# ================================================================================================
# STEP 2: Calculate Residuals (Errors)
# ================================================================================================

data['Residual'] = data['Actual_Saving'] - F0
print(f"\nStep 2: Residuals (Actual - Prediction)")
print(data[['Actual_Saving', 'Residual']])

# ================================================================================================
# STEP 3: Calculate Gradients and Hessians
# ================================================================================================

# For MSE loss: L = (y - ŷ)²
# Gradient: g = ∂L/∂ŷ = -2(y - ŷ)
# Hessian: h = ∂²L/∂ŷ² = 2

data['Gradient'] = -2 * data['Residual']  # g = -2 * residual
data['Hessian'] = 2  # h = 2 (constant for MSE)

print(f"\nStep 3: Gradients and Hessians")
print(data[['Residual', 'Gradient', 'Hessian']])

# ================================================================================================
# STEP 4: Build a Tree to Predict Residuals
# ================================================================================================

# Simplify: Suppose we have a tree that splits on Job_Value
# Split at Job_Value = 170000

left_split = data[data['Job_Value'] <= 170000]
right_split = data[data['Job_Value'] > 170000]

print(f"\nStep 4: Tree Split")
print(f"Left split (Job_Value ≤ 170000):")
print(left_split)
print(f"\nRight split (Job_Value > 170000):")
print(right_split)

# ================================================================================================
# STEP 5: Calculate Leaf Weights (with Regularization)
# ================================================================================================

def calculate_leaf_weight(gradients, hessians, lambda_reg=1):
    """
    Leaf weight = -Σ(gᵢ) / (Σ(hᵢ) + λ)
    Where:
    - gᵢ are gradients in the leaf
    - hᵢ are hessians in the leaf
    - λ is L2 regularization parameter
    """
    sum_g = np.sum(gradients)
    sum_h = np.sum(hessians)
    weight = -sum_g / (sum_h + lambda_reg)
    return weight

# Left leaf weight
left_gradients = left_split['Gradient'].values
left_hessians = left_split['Hessian'].values
left_weight = calculate_leaf_weight(left_gradients, left_hessians, lambda_reg=1)

# Right leaf weight
right_gradients = right_split['Gradient'].values
right_hessians = right_split['Hessian'].values
right_weight = calculate_leaf_weight(right_gradients, right_hessians, lambda_reg=1)

print(f"\nStep 5: Leaf Weights")
print(f"Left leaf weight: {left_weight:.4f}")
print(f"Right leaf weight: {right_weight:.4f}")

# ================================================================================================
# STEP 6: Make Predictions with Learning Rate (η)
# ================================================================================================

# Learning rate (shrinkage)
learning_rate = 0.3  # η

def predict_with_tree(data, split_value, left_weight, right_weight, learning_rate):
    """Make predictions using the tree"""
    predictions = []
    for job_value in data['Job_Value']:
        if job_value <= split_value:
            pred = learning_rate * left_weight
        else:
            pred = learning_rate * right_weight
        predictions.append(pred)
    return predictions

# Get tree predictions
data['Tree_Prediction'] = predict_with_tree(
    data, 170000, left_weight, right_weight, learning_rate
)

# Update predictions
data['New_Prediction'] = F0 + data['Tree_Prediction']

print(f"\nStep 6: Updated Predictions")
print(data[['Job_Value', 'Actual_Saving', 'Tree_Prediction', 'New_Prediction']])

# ================================================================================================
# STEP 7: Calculate New Residuals
# ================================================================================================

data['New_Residual'] = data['Actual_Saving'] - data['New_Prediction']

print(f"\nStep 7: New Residuals (after one tree)")
print(data[['Actual_Saving', 'New_Prediction', 'New_Residual']])

# ================================================================================================
# STEP 8: Calculate Total Loss
# ================================================================================================

def mse_loss(actual, predicted):
    return np.mean((actual - predicted) ** 2)

initial_loss = mse_loss(data['Actual_Saving'], [F0] * len(data))
new_loss = mse_loss(data['Actual_Saving'], data['New_Prediction'])

print(f"\nStep 8: Loss Reduction")
print(f"Initial MSE: {initial_loss:.2f}")
print(f"New MSE: {new_loss:.2f}")
print(f"Improvement: {initial_loss - new_loss:.2f} ({((initial_loss - new_loss)/initial_loss*100):.1f}%)")

# ================================================================================================
# STEP 9: The Full XGBoost Formula
# ================================================================================================

print("\n" + "="*70)
print("THE COMPLETE XGBOOST FORMULA")
print("="*70)

print("""
XGBoost Prediction After T Trees:

ŷᵢ = F₀ + η * Σₜ fₜ(xᵢ)

Where:
- F₀ = Initial prediction (mean of target)
- η = Learning rate (shrinkage parameter)
- fₜ(xᵢ) = Prediction from tree t for sample i

Tree Prediction (Leaf Weight):
wⱼ* = -Σᵢ∈leafⱼ gᵢ / (Σᵢ∈leafⱼ hᵢ + λ)

Where:
- gᵢ = Gradient (first derivative of loss)
- hᵢ = Hessian (second derivative of loss)
- λ = L2 regularization parameter

Regularization:
Ω(f) = γT + (1/2)λ||w||²

Where:
- T = Number of leaves
- γ = Penalty per leaf
- λ = L2 penalty on leaf weights

Objective Function (to minimize):
Obj = Σᵢ L(yᵢ, ŷᵢ) + Σₜ Ω(fₜ)
""")

Training Data:
   Job_Value  Labor_Hours  Actual_Saving
0     100000         1000          10000
1     200000         1500          15000
2     150000         1200          12000
3     250000         1800          18000
4     180000         1400          14000

Step 1: Initial Prediction F₀ = 13800.00

Step 2: Residuals (Actual - Prediction)
   Actual_Saving  Residual
0          10000   -3800.0
1          15000    1200.0
2          12000   -1800.0
3          18000    4200.0
4          14000     200.0

Step 3: Gradients and Hessians
   Residual  Gradient  Hessian
0   -3800.0    7600.0        2
1    1200.0   -2400.0        2
2   -1800.0    3600.0        2
3    4200.0   -8400.0        2
4     200.0    -400.0        2

Step 4: Tree Split
Left split (Job_Value ≤ 170000):
   Job_Value  Labor_Hours  Actual_Saving  Residual  Gradient  Hessian
0     100000         1000          10000   -3800.0    7600.0        2
2     150000         1200          12000   -1800.0    3600.0        2

Right split 

In [26]:
import numpy as np
import matplotlib.pyplot as plt

# Problem: Find x that minimizes f(x) = x² + 4x + 4
# This is (x+2)², so minimum is at x = -2

def f(x):
    return x**2 + 4*x + 4

def gradient(x):
    return 2*x + 4  # Derivative of f(x)

# Gradient descent
x = 5  # Start at x=5
learning_rate = 0.1
iterations = 20

print("Gradient Descent Steps:")
print(f"Start: x = {x}, f(x) = {f(x):.2f}")

for i in range(iterations):
    grad = gradient(x)
    x_new = x - learning_rate * grad  # Step in opposite direction of gradient
    x = x_new
    if i % 3 == 0:
        print(f"Iteration {i+1}: x = {x:.4f}, f(x) = {f(x):.4f}")

print(f"\nFinal: x = {x:.4f}, f(x) = {f(x):.4f}")
print(f"True minimum at x = -2, f(x) = 0")

Gradient Descent Steps:
Start: x = 5, f(x) = 49.00
Iteration 1: x = 3.6000, f(x) = 31.3600
Iteration 4: x = 0.8672, f(x) = 8.2208
Iteration 7: x = -0.5320, f(x) = 2.1550
Iteration 10: x = -1.2484, f(x) = 0.5649
Iteration 13: x = -1.6152, f(x) = 0.1481
Iteration 16: x = -1.8030, f(x) = 0.0388
Iteration 19: x = -1.8991, f(x) = 0.0102

Final: x = -1.9193, f(x) = 0.0065
True minimum at x = -2, f(x) = 0
